# GPU encoders on a free Kaggle GPU (spec 002 T017)

Fine-tunes the encoders that are too slow on CPU (the large ModernBERTs, the Longformers, and GatorTron-base at a
lower learning rate) on the training notes, then predicts the 240 review charts with
`python/scripts/11_ai_predict.py` from [health-transformation-lab-nlp](https://github.com/Health-Transformation-Lab/health-transformation-lab-nlp). This is the
work of the Databricks `encoder_gpu` job (notebook 08), for when Databricks Free Edition has no GPU to give.
Same script, same rungs and learning rates, and the same **synthetic** notes; nothing here touches real patient
data.

**Before you start (once):** a Kaggle account with a **verified phone number** (needed for GPUs and internet),
then **Accelerator → GPU T4 x2** and **Internet → On**. No token: every encoder is public on Hugging Face.

The GitHub workflow *Fine-tune the GPU encoders on a free Kaggle GPU* fills in the settings, pushes this
notebook and collects the results. By hand, run the cells in order; for the full run set `FULL_RUN = True` and
use **Save Version → Save & Run All (Commit)**.

With `SAVE_MODEL` set to a rung in `RUNGS` (e.g. `modernbert-base+cw`), that rung's fine-tuned weights are kept
in `/kaggle/working/model` as well, and the workflow publishes them as a Kaggle model after a full run (spec 020 US3).


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## Settings


In [ ]:
RUNGS = "modernbert-large,bioclinical-modernbert-large,longformer-base,clinical-longformer,gatortron-base"   # +cw: class-weighted
LR_OVERRIDES = "gatortron-base=2e-5"   # rung=rate, comma-separated; the others train at 5e-5
EPOCHS = 3
MAX_LEN = 256
TEST_NOTES = 10               # a test predicts this many notes after one epoch
BRANCH = "main"
COMMIT = ""                   # a commit of BRANCH to run; empty: its latest (the workflow pins it)
SAVE_MODEL = ""               # a rung in RUNGS whose fine-tuned weights to keep in /kaggle/working/model
FULL_RUN = False              # True: train EPOCHS and predict all 240 notes; set by the GitHub workflow


## Get the code


In [ ]:
import os
import sys

REPO = "/kaggle/temp/health-transformation-lab-nlp"      # outside /kaggle/working, so it isn't saved as output
os.makedirs("/kaggle/temp", exist_ok=True)
!git clone --quiet --depth 1 -b {BRANCH} https://github.com/Health-Transformation-Lab/health-transformation-lab-nlp {REPO}
if COMMIT:
    !git -C {REPO} fetch --quiet --depth 1 origin {COMMIT} && git -C {REPO} checkout --quiet FETCH_HEAD
!git -C {REPO} log -1 --date=short --format="code: {BRANCH} @ %h (%cd) %s"
!pip install --quiet -e "{REPO}/python[models]" "transformers>=4.48"
os.environ["DENTAL_TELEMETRY"] = "off"
sys.path.insert(0, f"{REPO}/python")
print("ready")


## Fine-tune and predict, one rung at a time

A rung that fails is reported and skipped; the others still run. Each rung's predictions are saved to
`/kaggle/working` (the **Output** tab) as soon as it finishes, so a run that hits Kaggle's time limit keeps them.


In [ ]:
import json
import shutil
import subprocess
import tempfile

from health_transformation_lab_nlp.rungs import output_tag

rungs = [r.strip() for r in RUNGS.split(",") if r.strip()]
lrs = dict(x.strip().split("=") for x in LR_OVERRIDES.split(",") if "=" in x)
run = ["--epochs", str(EPOCHS)] if FULL_RUN else ["--epochs", "1", "--limit", str(TEST_NOTES)]
work = tempfile.mkdtemp(prefix="encoder_gpu_", dir="/kaggle/temp")   # checkpoints: not kept as output
done, failed = [], []
if SAVE_MODEL and SAVE_MODEL not in rungs:
    rungs.append(SAVE_MODEL)
for rung in rungs:
    base, cw = rung.removesuffix("+cw"), rung.endswith("+cw")      # "modernbert-base+cw": class-weighted
    lr = float(lrs.get(rung, lrs.get(base, 5e-5)))
    cmd = [sys.executable, f"{REPO}/python/scripts/11_ai_predict.py", "--rung", base, "--max-len", str(MAX_LEN),
           "--lr", str(lr), "--grad-checkpoint", *run]   # T4: the large encoders don't fit otherwise (#172)
    if cw:
        cmd += ["--class-weight", "sqrt"]
    if rung == SAVE_MODEL:                      # spec 020 US3: the weights, for kaggle/publish_model.py
        cmd += ["--save-model", "/kaggle/working/model"]
    print(f"==== {rung} (lr {lr}, {'full run' if FULL_RUN else f'test: {TEST_NOTES} notes, 1 epoch'})", flush=True)
    if subprocess.run(cmd, cwd=work).returncode:
        print(f"==== {rung} failed: see above", flush=True)
        failed.append(rung)
        continue
    path = f"{REPO}/data/datasets/ai_predictions_{base}{output_tag(class_weight='sqrt' if cw else None, lr=lr)}.csv"
    shutil.copy(path, "/kaggle/working/")
    done.append(os.path.basename(path))
    if rung == SAVE_MODEL:                      # which predictions the kept weights made, for the model card
        with open("/kaggle/working/saved_model.json", "w") as f:
            json.dump({"rung": rung, "predictions": done[-1], "full_run": FULL_RUN}, f)
    print("saved", done[-1], flush=True)


## Result


In [ ]:
print("saved:", done, "| failed:", failed)
if failed:
    raise RuntimeError(f"encoders that failed: {failed} (the others' predictions are saved in /kaggle/working)")


## Back in the repo

The workflow opens a pull request with a full run's files (spec 016); after the merge, **Load results** puts them
in `gold_perio_bench.fact_prediction` with `source_code = kaggle`. By hand: put the downloaded
`ai_predictions_<rung>.csv` files in `data/datasets/`, then
`cd python && python scripts/28_ladder_vs_truth.py && python scripts/35_readme_results.py`.
